In [1]:
import glob
import os

def concatenate_clump_files():
    # Find all files matching the pattern
    pattern = "output_00002/clump_00002.txt0000*"
    files = glob.glob(pattern)
    
    if not files:
        print(f"No files found matching pattern: {pattern}")
        return
    
    # Sort files to ensure consistent ordering
    files.sort()
    print(f"Found {len(files)} files: {files}")
    
    output_file = "clump_all.txt"
    header_written = False
    
    with open(output_file, 'w') as outfile:
        for i, filename in enumerate(files):
            print(f"Processing {filename}...")
            
            with open(filename, 'r') as infile:
                lines = infile.readlines()
                
                # Skip empty files
                if not lines:
                    continue
                
                # Write header only from the first file
                if not header_written and lines:
                    outfile.write(lines[0])  # Write header
                    header_written = True
                
                # Write data lines (skip header for all files)
                for line in lines[1:]:
                    outfile.write(line)
    
    print(f"Successfully concatenated {len(files)} files into {output_file}")
    
    # Show summary
    with open(output_file, 'r') as f:
        total_lines = sum(1 for _ in f)
    print(f"Clump concatenated output file contains {total_lines} lines (including 1 header)")

if __name__ == "__main__":
    concatenate_clump_files()




Found 4 files: ['output_00002/clump_00002.txt00001', 'output_00002/clump_00002.txt00002', 'output_00002/clump_00002.txt00003', 'output_00002/clump_00002.txt00004']
Processing output_00002/clump_00002.txt00001...
Processing output_00002/clump_00002.txt00002...
Processing output_00002/clump_00002.txt00003...
Processing output_00002/clump_00002.txt00004...
Successfully concatenated 4 files into clump_all.txt
Clump concatenated output file contains 24 lines (including 1 header)


In [3]:
import filecmp

def simple_diff():
    file1 = "clump_all.txt"
    file2 = "clump-ref.txt"
    
    # Quick check if files are identical
    if filecmp.cmp(file1, file2):
        print("OK! Clump Files are identical => Passed")
        return
    
    print("ERROR clump Files differ from reference clump-ref.dat")
    
    # Show first differing line
    with open(file1) as f1, open(file2) as f2:
        for i, (line1, line2) in enumerate(zip(f1, f2), 1):
            if line1 != line2:
                print(f"First difference at line {i}:")
                print(f"  {file1}: {line1.rstrip()}")
                print(f"  {file2}: {line2.rstrip()}")
                break
        else:
            # Files have different lengths
            print("Files have different lengths")

if __name__ == "__main__":
    simple_diff()


OK! Clump Files are identical => Passed


In [5]:
import numpy as np
# Read the file using numpy.loadtxt, skipping the header row
data = np.loadtxt("clump_all.txt", skiprows=1)

# Extract the columns you need
# Based on your file format:
# Column indices: 0=index, 1=halo, 2=lev, 3=parent, 4=ncell, 5=peak_x, 6=peak_y, 7=peak_z, 8=rho-, 9=rho+, 10=rho_av, 11=mass_cl, 12=relevance
ncell_output_array = data[:, 4]      # ncell column (5th column, index 4)
mass_cl_output_array = data[:, 11]   # mass_cl column (12th column, index 11)

print("ncell values:", ncell_output_array)
print("mass_cl values:", mass_cl_output_array)

ncell values: [ 37.  21.   5.  53.  12.  14.   5.   8.   5. 150.  21.  11.  19.  26.
  13.  53.  18.  20.  17.  13.   7.   8.   7.]
mass_cl values: [0.00385275 0.00158942 0.00039397 0.004412   0.00099431 0.00124981
 0.00040881 0.00064334 0.00039178 0.01892537 0.00173548 0.00077794
 0.00159811 0.00162409 0.00100696 0.00557033 0.00157329 0.00123927
 0.00115691 0.00085357 0.00049351 0.00056489 0.00055667]


In [2]:
import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt
import visu_ramses
from matplotlib.colors import LogNorm

fig = plt.figure(figsize=(12, 3.75))
axes = fig.subplots(nrows=1, ncols=3)

# Load RAMSES output
data = visu_ramses.load_snapshot(2,read_hydro=False)
xp = data["particle"]["position_x"]
yp = data["particle"]["position_y"]
zp = data["particle"]["position_z"]
mp = data["particle"]["mass"]

im = axes[0].hist2d(xp,yp,weights=mp,bins=128,range=[[0, 1], [0, 1]],norm=LogNorm(vmin=8e-6,vmax=8e-4),cmap='bone',edgecolor='face')
im = axes[1].hist2d(xp,zp,weights=mp,bins=128,range=[[0, 1], [0, 1]],norm=LogNorm(vmin=8e-6,vmax=8e-4),cmap='bone',edgecolor='face')
im = axes[2].hist2d(yp,zp,weights=mp,bins=128,range=[[0, 1], [0, 1]],norm=LogNorm(vmin=8e-6,vmax=8e-4),cmap='bone',edgecolor='face')
#plt.colorbar(im[3], ax=axes[2])
for ax in axes:
    ax.axis('equal')
    ax.set_xlim([0,1])
    ax.set_ylim([0,1])
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[1].set_xlabel('x')
axes[1].set_ylabel('z')
axes[2].set_xlabel('y')
axes[2].set_ylabel('z')

fig.savefig("cosmo.pdf", bbox_inches="tight")

to_check = data["particle"]
to_check['time'] = data["data"]["time"]

visu_ramses.check_solution(to_check, 'cosmo')


file descriptor not found: output_00002/hydro_file_descriptor.txt
Processing 4 files in output_00002
 25% : read     524288 cells
 50% : read    1048576 cells
 75% : read    1572864 cells
Total number of cells loaded: 2097152
Total particles loaded: 2097152


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


The current and reference solutions do not have the same variables
